In [6]:
from pathlib import Path
import matplotlib.pyplot as plt
import json
import pandas as pd
import numpy as np
import lightkurve as lk
import warnings
from tqdm.auto import tqdm

import time
from astropy.utils.data import conf

warnings.filterwarnings("ignore")

### 1. Prepare Data Directories
These will be used by the custom DataLoader to load lightcurve data

In [ ]:
# Creating directories for data and lightcurves
data_path = Path("data")
lightcurves_path = data_path / "lightcurves"

lightcurves_path.mkdir(parents = True, exist_ok = True)

# Save processed labels to the data directory
labels_path = data_path / "processed_labels.csv"

consensus = pd.read_csv(labels_path)

print(f"Successfully loaded {len(consensus)} stars")
print(consensus.head())

Successfully loaded 2694 stars
      TIC_ID  final_label  Period (days)   Epoch (BJD)
0   90919952            0       4.138120  2.458660e+06
1  262457587            0       0.821960  2.460123e+06
2   50120414            1       3.581083  2.459932e+06
3   17457400            1       1.276150  2.459523e+06
4  157568289            0       1.836150  2.459251e+06


### 2. Transform and Load Data

In [ ]:
conf.remote_timeout = 30

# target tensor length to prevent shape mismatch errors
target_length = 20000
successful_stars = 0

# loop over all the stars in the list
for index, row in tqdm(consensus.iterrows(), total = len(consensus), desc = "Processing Stars"):
    tic_id = int(row['TIC_ID'])
    label = row['final_label']

    save_path = lightcurves_path / f"TIC_{tic_id}.npy"

    if save_path.is_file():
        successful_stars += 1
        continue

    try:
        # download the raw data
        search_result = lk.search_lightcurve(f"TIC {tic_id}", mission = "TESS", author = "SPOC")
        if len(search_result) == 0:
            tqdm.write(f"TIC {tic_id} download failed.")
            continue

        lc = search_result[0].download()

        # Transform data
        lc_cleaned = lc.remove_nans().flatten(window_length = 901)

        # Calculate relative tine starting from day 0
        rel_time = lc_cleaned.time.value - lc_cleaned.time.value[0]

        # Create and apply a mask to remove day 6 and 18.5 data (momentum dumps, refer to notebook_01)
        mask = ~(((rel_time > 5.5) & (rel_time < 6.5)) | ((rel_time > 18.0) & (rel_time < 19.5)))
        lc_cleaned = lc_cleaned[mask]

        """
        # Find orbital period using BLS
        periodogram = lc_cleaned.to_periodogram(method = "bls", minimum_period = 1, maximum_period = 15)
        best_period = periodogram.period_at_max_power
        first_epoch = periodogram.transit_time_at_max_power

        # Phase-fold the data based on calculated orbital period
        lc_folded = lc_cleaned.fold(period = best_period, epoch_time = first_epoch)
        """
        
        catalog_period = row['Period (days)']
        catalog_epoch  = row['Epoch (BJD)']
    
        if pd.notna(catalog_period) and pd.notna(catalog_epoch):
            best_period, first_epoch = catalog_period, catalog_epoch
        else:
            # Note: manually calculate orbital period using BLS (refer to notebook_01) if exact numbers havent been published
            periodogram = lc_cleaned.to_periodogram(method = "bls", minimum_period = 1, maximum_period = 15)
            best_period = periodogram.period_at_max_power
            first_epoch = periodogram.transit_time_at_max_power
        
        lc_folded = lc_cleaned.fold(period = best_period, epoch_time = first_epoch)

        # Extracting raw number and shaping tensor data
        raw_flux = lc_folded.flux.value
        
        # Remove the Astropy 'MaskedNDArray' wrap
        if hasattr(raw_flux, 'unmasked'):
            flux_array = raw_flux.unmasked
        else:
            flux_array = raw_flux
            
        flux_array = np.array(flux_array, dtype = np.float32)
        
        if len(flux_array) > target_length:
            flux_array = flux_array[:target_length] 
        else:
            pad_left_size = (target_length - len(flux_array)) // 2
            pad_right_size = target_length - len(flux_array) - pad_left_size
            flux_array = np.pad(flux_array, (pad_left_size, pad_right_size), mode='wrap')
        
        np.save(save_path, flux_array)
        
        successful_stars += 1

    except Exception as e:
        tqdm.write(f"TIC {tic_id} failed during processing: {e}")

print(f"{successful_stars}/{len(consensus)} Stars successfully saved")
    

Processing Stars:   0%|          | 0/2694 [00:00<?, ?it/s]

TIC 90919952 download failed.
TIC 262457587 download failed.
TIC 77552382 download failed.
TIC 469523619 download failed.
TIC 77258176 download failed.
TIC 132320310 download failed.
TIC 252610172 failed during processing: Error in reading Data product /Users/Vallabh/.cache/lightkurve/mastDownload/TESS/tess2024085201119-s0077-0000000252610172-0272-s/tess2024085201119-s0077-0000000252610172-0272-s_lc.fits of type TessLightCurve .
This file may be corrupt due to an interrupted download. Please remove it from your disk and try again.
TIC 126772538 download failed.
TIC 365712006 download failed.
TIC 19269186 download failed.
TIC 405862830 download failed.
TIC 453289469 download failed.
TIC 334125713 download failed.
TIC 435167167 download failed.
TIC 374177909 download failed.
TIC 146208651 download failed.
TIC 314338017 download failed.
TIC 263313709 download failed.
TIC 373729723 download failed.
TIC 207334420 download failed.
TIC 36646052 download failed.
TIC 301623068 download failed.


TIC 190981760 download failed.
TIC 307398951 download failed.
TIC 74484886 download failed.
TIC 41067039 download failed.
TIC 200465781 download failed.
TIC 237019002 download failed.
TIC 75417968 download failed.
TIC 193081156 download failed.
TIC 427715152 download failed.
TIC 139285736 download failed.
TIC 112037203 download failed.
2284/2694 Stars successfully saved


In [ ]:
import numpy as np
from pathlib import Path
from tqdm import tqdm

lightcurves_dir = Path("data/lightcurves")
target_length = 20000
center_idx = target_length // 2

files = list(lightcurves_dir.glob("*.npy"))
success_count = 0
failed_files = []

for f in tqdm(files, desc="Re-centering"):
    try:
        # Load the file array
        arr = np.load(f)
        
        # Calculate shift and roll
        shift = center_idx - int(np.argmin(arr))
        rolled_arr = np.roll(arr, shift)
        
        # Save the file back to disk
        np.save(f, rolled_arr)
        success_count += 1
        
    except Exception as e:
        tqdm.write(f"\n[WARNING] Failed to process {f.name}: {e}")
        failed_files.append(f.name)

print(f"\nRe-centering complete.")
print(f"Successfully processed: {success_count}/{len(files)} files.")

if failed_files:
    print(f"Failed files ({len(failed_files)}): {failed_files}")

Re-centering: 100%|███████████████████████████████████████████████████████████████████| 2263/2263 [00:00<00:00, 2282.55it/s]


Re-centering complete.
Successfully processed: 2263/2263 files.


In [1]:
import numpy as np
from pathlib import Path

lightcurves_dir = Path("data/lightcurves")
suspicious = []

for f in lightcurves_dir.glob("*.npy"):
    arr = np.load(f)
    if abs(arr.mean() - 1.0) > 0.5 or arr.std() < 1e-6:
        suspicious.append((f.name, float(arr.mean()), float(arr.std())))

print(f"Suspicious files: {len(suspicious)} / {len(list(lightcurves_dir.glob('*.npy')))}")
for name, m, s in suspicious[:15]:
    print(f"  {name}: mean={m:.4f}, std={s:.6f}")

Suspicious files: 0 / 2263
